
**Name:** Reid Sendroff

In [ ]:
%pip install -q datasets tokenizers sacrebleu
!python -m spacy download en_core_web_md -q

# Problem 1

In [ ]:
import spacy
import numpy as np

In [ ]:
nlp = spacy.load("en_core_web_md")

In [ ]:
prince_vec = nlp("prince").vector
man_vec = nlp("man").vector
woman_vec = nlp("woman").vector
comb_vec = prince_vec - man_vec + woman_vec

print(comb_vec.shape)

In [ ]:
query_vector = comb_vec.reshape(1, -1)
keys, rows, cosine_scores = nlp.vocab.vectors.most_similar(query_vector, n=20)

top_simvecs = []
for key, row, score in zip(keys[0], rows[0], cosine_scores[0]):
  word = nlp.vocab.strings[key]
  if word.lower() not in {"prince", "man", "woman"} and word.isalpha():
    top_simvecs.append((float(score), word, nlp.vocab.vectors.data[row]))
  if len(top_simvecs) == 5:
    break

for number, (score, word, vector) in enumerate(top_simvecs, start=1):
  print(f"{number}. {word.upper()} (cosine similarity={score:.4f})")
  print("   first 10 values:", np.round(vector[:10], 4))

### Problem 1 findings

The closest words are `HAIRDRESSER`, `GAL`, and `ACTRESS` (0.7345), followed by `STUNNER` and `STARLET` (0.6850). The ties occur because spaCy stores identical vectors for those words. I excluded `prince`, `man`, and `woman` from the results.


# Problem 2

### Training setup

I used 20,000 pairs from `Helsinki-NLP/opus_books`, with French as the input and English as the target. I reserved 2,000 pairs for testing and 1,800 for validation. The encoder-decoder Transformer uses two layers per side, four attention heads, 128-dimensional embeddings, and a shared 10,000-token BPE vocabulary.


In [ ]:
from pathlib import Path
from collections import namedtuple
import random
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader
import tokenizers
import pandas as pd
from IPython.display import display

In [ ]:
SEED = 42
MAX_PAIRS = 20_000
TEST_FRACTION = 0.10
VALID_FRACTION_OF_TRAIN = 0.10
MAX_LENGTH = 40
VOCAB_SIZE = 10_000
BATCH_SIZE = 64
EMBED_DIM = 128
NUM_HEADS = 4
NUM_LAYERS = 2
DROPOUT = 0.1
N_EPOCHS = 8

In [ ]:
torch.manual_seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

all_pairs = load_dataset("Helsinki-NLP/opus_books", "en-fr", split="train")
all_pairs = all_pairs.shuffle(seed=SEED).select(range(min(MAX_PAIRS, len(all_pairs))))
test_split = all_pairs.train_test_split(test_size=TEST_FRACTION, seed=SEED)
train_and_valid_set = test_split["train"]
nmt_test_set = test_split["test"]

valid_split = train_and_valid_set.train_test_split(
  test_size=VALID_FRACTION_OF_TRAIN,
  seed=SEED,
)
nmt_train_set = valid_split["train"]
nmt_valid_set = valid_split["test"]

print(f"Train: {len(nmt_train_set):,}; validation: {len(nmt_valid_set):,}; test: {len(nmt_test_set):,}")

In [ ]:
def training_text():
  for example in nmt_train_set:
    pair = example["translation"]
    yield pair["fr"]
    yield pair["en"]

nmt_tokenizer = tokenizers.Tokenizer(tokenizers.models.BPE(unk_token="<unk>"))
nmt_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
nmt_tokenizer.enable_truncation(max_length=MAX_LENGTH)
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
trainer = tokenizers.trainers.BpeTrainer(
  vocab_size=VOCAB_SIZE,
  special_tokens=["<pad>", "<unk>", "<s>", "</s>"],
)
nmt_tokenizer.train_from_iterator(training_text(), trainer)

fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]

class NmtPair(namedtuple("NmtPairBase", fields)):
  def to(self, device):
    return NmtPair(*(tensor.to(device) for tensor in self))


def nmt_collate_fn(batch):
  french = [example["translation"]["fr"] for example in batch]
  english = [f"<s> {example['translation']['en']} </s>" for example in batch]
  source_encodings = nmt_tokenizer.encode_batch(french)
  target_encodings = nmt_tokenizer.encode_batch(english)

  source_ids = torch.tensor([item.ids for item in source_encodings], dtype=torch.long)
  target_ids = torch.tensor([item.ids for item in target_encodings], dtype=torch.long)
  source_mask = torch.tensor([item.attention_mask for item in source_encodings], dtype=torch.long)
  target_mask = torch.tensor([item.attention_mask for item in target_encodings], dtype=torch.long)

  inputs = NmtPair(source_ids, source_mask, target_ids[:, :-1], target_mask[:, :-1])
  labels = target_ids[:, 1:]
  return inputs, labels


nmt_train_loader = DataLoader(
  nmt_train_set,
  batch_size=BATCH_SIZE,
  shuffle=True,
  collate_fn=nmt_collate_fn,
)
nmt_valid_loader = DataLoader(
  nmt_valid_set,
  batch_size=BATCH_SIZE,
  collate_fn=nmt_collate_fn,
)

In [ ]:
class PositionalEmbedding(nn.Module):
  def __init__(self, max_length, embed_dim, dropout=DROPOUT):
    super().__init__()
    self.pos_embed = nn.Parameter(torch.randn(max_length, embed_dim) * 0.02)
    self.dropout = nn.Dropout(dropout)

  def forward(self, embeddings):
    return self.dropout(embeddings + self.pos_embed[:embeddings.size(1)])


class NmtTransformer(nn.Module):
  def __init__(self, vocab_size, max_length, embed_dim=EMBED_DIM, pad_id=0,
               num_heads=NUM_HEADS, num_layers=NUM_LAYERS, dropout=DROPOUT):
    super().__init__()
    self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
    self.pos_embed = PositionalEmbedding(max_length, embed_dim, dropout)
    self.transformer = nn.Transformer(
      d_model=embed_dim,
      nhead=num_heads,
      num_encoder_layers=num_layers,
      num_decoder_layers=num_layers,
      dropout=dropout,
      batch_first=True,
    )
    self.output = nn.Linear(embed_dim, vocab_size)

  def forward(self, pair):
    source_embeddings = self.pos_embed(self.embed(pair.src_token_ids))
    target_embeddings = self.pos_embed(self.embed(pair.tgt_token_ids))
    source_pad_mask = ~pair.src_mask.bool()
    target_pad_mask = ~pair.tgt_mask.bool()

    target_length = pair.tgt_token_ids.size(1)
    causal_mask = torch.triu(
      torch.ones(
        (target_length, target_length),
        device=source_embeddings.device,
        dtype=torch.bool,
      ),
      diagonal=1,
    )

    decoded = self.transformer(
      source_embeddings,
      target_embeddings,
      tgt_mask=causal_mask,
      src_key_padding_mask=source_pad_mask,
      memory_key_padding_mask=source_pad_mask,
      tgt_key_padding_mask=target_pad_mask,
    )
    return self.output(decoded).permute(0, 2, 1)

In [ ]:
def evaluate(model, data_loader, loss_fn):
  model.eval()
  total_loss = 0
  total_correct = 0
  total_tokens = 0

  with torch.no_grad():
    for inputs, labels in data_loader:
      inputs = inputs.to(device)
      labels = labels.to(device)
      logits = model(inputs)
      total_loss += loss_fn(logits, labels).item()

      non_padding = labels.ne(0)
      predictions = logits.argmax(dim=1)
      total_correct += (predictions.eq(labels) & non_padding).sum().item()
      total_tokens += non_padding.sum().item()

  return total_loss / len(data_loader), total_correct / total_tokens

In [ ]:
def train_model(model, train_loader, valid_loader, epochs):
  criterion = nn.CrossEntropyLoss(ignore_index=0)
  optimizer = torch.optim.NAdam(model.parameters())
  history = {"train_loss": [], "valid_loss": [], "valid_token_accuracy": []}

  for epoch in range(epochs):
    model.train()
    total_train_loss = 0

    for inputs, labels in train_loader:
      inputs = inputs.to(device)
      labels = labels.to(device)
      optimizer.zero_grad()

      logits = model(inputs)
      loss = criterion(logits, labels)
      loss.backward()
      optimizer.step()
      total_train_loss += loss.item()

    valid_loss, valid_token_accuracy = evaluate(model, valid_loader, criterion)
    train_loss = total_train_loss / len(train_loader)
    history["train_loss"].append(train_loss)
    history["valid_loss"].append(valid_loss)
    history["valid_token_accuracy"].append(valid_token_accuracy)

    print(
      f"Epoch {epoch + 1}/{epochs}: train loss={train_loss:.4f}, "
      f"valid loss={valid_loss:.4f}, valid token accuracy={valid_token_accuracy:.2%}"
    )

  return history

In [ ]:
model = NmtTransformer(nmt_tokenizer.get_vocab_size(), MAX_LENGTH).to(device)
history = train_model(model, nmt_train_loader, nmt_valid_loader, N_EPOCHS)

artifact_dir = Path("assign07_artifacts")
artifact_dir.mkdir(exist_ok=True)
model_path = artifact_dir / "french_to_english_transformer.pt"
tokenizer_path = artifact_dir / "french_to_english_tokenizer.json"
model_config = {
  "vocab_size": nmt_tokenizer.get_vocab_size(),
  "max_length": MAX_LENGTH,
  "embed_dim": EMBED_DIM,
  "pad_id": 0,
  "num_heads": NUM_HEADS,
  "num_layers": NUM_LAYERS,
  "dropout": DROPOUT,
}

torch.save({"model_state_dict": model.state_dict(), "model_config": model_config}, model_path)
nmt_tokenizer.save(str(tokenizer_path))
print("Saved model:", model_path)
print("Saved tokenizer:", tokenizer_path)

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, N_EPOCHS + 1)
plt.plot(epochs, history["train_loss"], marker="o", label="Training loss")
plt.plot(epochs, history["valid_loss"], marker="o", label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("French-to-English transformer loss")
plt.xticks(epochs)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
def translate(model, french_text, max_length=MAX_LENGTH):
  model.eval()
  generated_english = ""
  eos_id = nmt_tokenizer.token_to_id("</s>")

  for index in range(max_length - 1):
    example = {"translation": {"fr": french_text, "en": generated_english}}
    inputs, _ = nmt_collate_fn([example])
    with torch.no_grad():
      token_id = int(model(inputs.to(device)).argmax(dim=1)[0, index].item())

    if token_id == eos_id:
      break

    token = nmt_tokenizer.id_to_token(token_id)
    if token not in {"<pad>", "<s>", "<unk>"}:
      generated_english += (" " if generated_english else "") + token

  return generated_english


short_test_examples = [
  example
  for example in nmt_test_set.shuffle(seed=SEED)
  if len(example["translation"]["fr"].split()) <= 10
][:5]

google_translations = [
  "Julien greeted with respect.",
  "—Shut your mouth, for God’s sake!…",
  "Admittedly, the government was doing a lot—but not enough!",
  "-- Hold fast, Billy, for he looks vicious.",
  "At that moment, Cyrus Smith had other thoughts on his mind.",
]

comparison_rows = []
for number, (example, google_english) in enumerate(zip(short_test_examples, google_translations), start=1):
  french = example["translation"]["fr"]
  comparison_rows.append({
    "Sentence": number,
    "French test input": french,
    "Reference English": example["translation"]["en"],
    "Model English": translate(model, french),
    "Google Translate English": google_english,
  })

comparison = pd.DataFrame(comparison_rows)
pd.set_option("display.max_colwidth", None)
display(comparison)

### Translation findings

1. Google preserves the first sentence, while my model changes the subject and action.
2. Google preserves the command; my model changes its meaning.
3. Google is close to the reference; my model produces a different sentence.
4. Google preserves the warning to Billy; my model loses the meaning.
5. Google matches the reference's idea; my model changes it.

Training loss fell from 6.0431 to 4.1241, while validation token accuracy reached 22.12%. The model learned some structure, but this small eight-epoch run is not enough for strong translation quality.


# Problem 3

In [ ]:
import gc
import warnings

if torch.cuda.is_available():
  gpu_tensors = []
  with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    for tensor in list(gc.get_objects()):
      try:
        if isinstance(tensor, torch.Tensor) and tensor.is_cuda:
          gpu_tensors.append({
            "Shape": tuple(tensor.shape),
            "Dtype": str(tensor.dtype).replace("torch.", ""),
            "Elements": tensor.numel(),
            "Memory (MiB)": tensor.numel() * tensor.element_size() / 2**20,
          })
      except Exception:
        pass

  gpu_table = pd.DataFrame(gpu_tensors).sort_values(
    "Memory (MiB)",
    ascending=False,
  ).reset_index(drop=True)
  gpu_table.index = range(1, len(gpu_table) + 1)
  gpu_table.index.name = "Tensor"
  total_tensor_memory = gpu_table["Memory (MiB)"].sum()

  print(f"CUDA tensor objects found: {len(gpu_table)}")
  display(gpu_table.style.format({"Memory (MiB)": "{:.4f}"}))
  print(f"Total memory represented by these tensors: {total_tensor_memory:.2f} MiB")
  print(f"CUDA allocated before cleanup: {torch.cuda.memory_allocated() / 2**20:.2f} MiB")
  print(f"CUDA reserved before cleanup: {torch.cuda.memory_reserved() / 2**20:.2f} MiB")

  del gpu_tensors
  del tensor
  del model
  gc.collect()
  torch.cuda.empty_cache()

  print(f"CUDA allocated after cleanup: {torch.cuda.memory_allocated() / 2**20:.2f} MiB")
  print(f"CUDA reserved after cleanup: {torch.cuda.memory_reserved() / 2**20:.2f} MiB")
elif torch.backends.mps.is_available():
  mps_tensors = []
  for tensor in list(gc.get_objects()):
    try:
      if isinstance(tensor, torch.Tensor) and tensor.is_mps:
        mps_tensors.append({
          "Shape": tuple(tensor.shape),
          "Dtype": str(tensor.dtype).replace("torch.", ""),
          "Elements": tensor.numel(),
          "Memory (MiB)": tensor.numel() * tensor.element_size() / 2**20,
        })
    except Exception:
      pass

  mps_table = pd.DataFrame(mps_tensors)
  display(mps_table)
  del mps_tensors
  del model
  gc.collect()
  torch.mps.empty_cache()
else:
  print("No CUDA or MPS accelerator is available.")

### GPU-memory findings

The table lists every live CUDA tensor's shape, data type, element count, and memory. After deleting the model references, running garbage collection, and clearing the CUDA cache, allocated memory fell from 56.01 MiB to 17.25 MiB and reserved memory fell from 814.00 MiB to 118.00 MiB.


# Problem 4

In [ ]:
from sacrebleu.metrics import BLEU

checkpoint = torch.load(model_path, map_location=device)
nmt_tokenizer = tokenizers.Tokenizer.from_file(str(tokenizer_path))
reloaded_model = NmtTransformer(**checkpoint["model_config"]).to(device)
reloaded_model.load_state_dict(checkpoint["model_state_dict"])
reloaded_model.eval()

bleu = BLEU(force=True)
first_50 = nmt_test_set.select(range(min(50, len(nmt_test_set))))
hypotheses_50 = [translate(reloaded_model, row["translation"]["fr"]) for row in first_50]
references_50 = [row["translation"]["en"] for row in first_50]
score_50 = bleu.corpus_score(hypotheses_50, [references_50])

hypotheses_full = []
references_full = []
for number, row in enumerate(nmt_test_set, start=1):
  hypotheses_full.append(translate(reloaded_model, row["translation"]["fr"]))
  references_full.append(row["translation"]["en"])
  if number % 250 == 0:
    print(f"Translated {number:,} of {len(nmt_test_set):,} test sentences")

score_full = bleu.corpus_score(hypotheses_full, [references_full])
print()
print(f"BLEU on 50 test sentences: {score_50.score:.2f}")
print(f"BLEU on the full test set ({len(nmt_test_set):,} sentences): {score_full.score:.2f}")

### BLEU interpretation

BLEU was **1.77** on 50 sentences and **1.32** on all 2,000 test sentences. The full-test score is more stable, and both low scores match the weak sample translations. BLEU measures n-gram overlap with a brevity penalty, so it does not fully measure fluency or meaning.
